# ap2103_extract_other_data

## Import Data From Database and Export as CSV

### Summary

This notebook connects to the target database, retrieves the relevant metadata tables, and exports them as CSV files. All steps—from establishing the connection to saving the outputs—are organized and documented for clarity and reuse.

### Notebook Structure

1. **Imports & Definitions**  
    Initialization of required libraries, configuration parameters, and helper variables.

2. **Function Definitions**  
    Implementation of reusable functions for connecting to the database, querying tables, and handling data transformations.

3. **Load Data**  
    Execution of queries to import all necessary metadata tables into the notebook environment.

4. **Save as CSV**  
    Export of the loaded tables into CSV files, including optional logging and overwrite handling.

### Findings

- All required tables were successfully loaded without errors.
- No irregularities or unexpected values were detected in the retrieved data.

### Open Tasks

- Integrate the data-loading and export function into the existing module for improved reusability and automation.

In [ ]:
import os
import pandas as pd

from dotenv import load_dotenv
from sshtunnel import SSHTunnelForwarder
from datetime import datetime
from modules.data_loading import load_energy_community_data, load_all_metering_points_in_energy_community_data, get_postgres_engine

### 01 Imports & Definitionen

In [ ]:
path_to_local_data = "../../local_data/"
date = datetime.now()   # current date
date = date.strftime("%Y-%m-%d")  # date in YYYY-MM-DD format

use_csv = 0  # 1 = use CSV files, 0 = use database

# tables to export and filenames
descriptions = {
    "Data extraction date": date,
    "org_id": "Organisation ID",
    "time": "Measurement timestamp",
    "metering_points_cnt": "Number of metering points in the REC",
    "consumer_count": "Number of consumers in the REC",
    "generator_count": "Number of producers in the REC",
    "sum_wt_meas_cons": "Sum of measured consumption weighted by participation factor",
    "sum_comm_pot": "Sum of community potential",
    "sum_comm_cov": "Sum of community coverage",
    "sum_wt_meas_gen": "Sum of measured generation weighted by participation factor",
    "sum_wt_surp_gen": "Sum of residual surplus weighted by participation factor"
}

### 02 Funktion

In [ ]:
def load_master_data(sql_engine) -> dict:
    """Loads all data from database and returns dict of DataFrames"""

    tables = [
        "v_proj_metering_point",
        "v_proj_mp_p",
        "v_proj_org_mp",
        "v_proj_org_p",
        "v_proj_participant",
        "v_proj_organization"
    ]

    dfs = {}

    for table in tables:
        sql_query = f"SELECT * FROM {table}"
        df = pd.read_sql(sql_query, sql_engine)
        dfs[f"df_{table}"] = df

    return dfs

### 03 load data

In [ ]:
load_dotenv()
ssh_host = os.getenv("SSH_HOST")
ssh_port = int(os.getenv("SSH_PORT"))
ssh_user = os.getenv("SSH_USER")
ssh_pw = os.getenv("SSH_PASSWORD")

postgres_server_ip = os.getenv("POSTGRES_SERVER_IP")
postgres_port = int(os.getenv("POSTGRES_PORT"))

In [ ]:
#load_master_data

database = "v_proj_organization"

if use_csv == 0:

    load_dotenv()
    ssh_host = os.getenv("SSH_HOST")
    ssh_port = int(os.getenv("SSH_PORT"))
    ssh_user = os.getenv("SSH_USER")
    ssh_pw = os.getenv("SSH_PASSWORD")

    postgres_server_ip = os.getenv("POSTGRES_SERVER_IP")
    postgres_port = int(os.getenv("POSTGRES_PORT"))

    with SSHTunnelForwarder(
        (ssh_host, ssh_port),
        ssh_username=ssh_user,
        ssh_password=ssh_pw,
        remote_bind_address=(ssh_host, postgres_port),
        local_bind_address=(postgres_server_ip, postgres_port)
    ) as tunnel:
        df_v_proj_metering_point, df_v_proj_mp_p, df_v_proj_org_mp, df_v_proj_org_p, df_v_proj_participant, df_v_proj_organization=load_master_data(sql_engine=get_postgres_engine())

In [ ]:
#example dataset

df_v_proj_organization.head()

### 10 Export to csv

In [ ]:
date = datetime.now().strftime("%Y%m%d")

dataframes = {
    "df_v_proj_metering_point_"+date: df_v_proj_metering_point,
    "df_v_proj_mp_p_"+date: df_v_proj_mp_p,
    "df_v_proj_org_mp_"+date: df_v_proj_org_mp,
    "df_v_proj_org_p_"+date: df_v_proj_org_p,
    "df_v_proj_participant_"+date: df_v_proj_participant,
    "df_v_proj_organization_"+date: df_v_proj_organization
}

for name, df in dataframes.items():
    filename = f"data/processed_data/{name}.csv"
    df.to_csv(filename, index=False)